# Name That LLM：共享基线研究入口

先按 README 用项目 .venv 注册 src/ntl。核心输入、分折、模型、OOF 和提交逻辑均由公共包提供。
此研究入口会运行一次单折检查和正式五折，保存独立产物，再可选全量训练预测。
Kaggle 上传生成的 99_submission.ipynb，不上传本研究 Notebook。


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from ntl.data import resolve_data_dir, KAGGLE_DATA, REQUIRED_DATA_FILES
from ntl.models import make_model, RANDOM_SEED

DATA_DIR = resolve_data_dir(os.environ.get('NAME_THAT_LLM_DATA_DIR'))
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print(f'Data: {DATA_DIR.resolve()}')
print(f'Output: {OUTPUT_DIR.resolve()}')


In [ ]:
from ntl.labels import OFFICIAL_MODEL_LABELS, class_order
from ntl.data import load_competition_data
train, test, sample_submission, data_summary = load_competition_data(DATA_DIR)
labels = list(class_order)
print(json.dumps(data_summary, ensure_ascii=False, indent=2))


In [ ]:
from datetime import datetime, timezone
from ntl.experiments import run_experiment
from ntl.metrics import scores_summary

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'pyproject.toml').is_file() and (p / 'experiments/folds/folds_v1.csv').is_file())
RUN_ID = 'research_baseline_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
run_experiment(RUN_ID + '_smoke', mode='smoke', data_dir=DATA_DIR, project_root=PROJECT_ROOT)
RUN_DIR = run_experiment(RUN_ID, mode='formal', data_dir=DATA_DIR,
                         project_root=PROJECT_ROOT, smoke_run=RUN_ID + '_smoke')
fold_results = pd.read_csv(RUN_DIR / 'fold_metrics.csv')
class_results = pd.read_csv(RUN_DIR / 'class_recall.csv')
oof = pd.read_parquet(RUN_DIR / 'oof_predictions.parquet')
print(scores_summary(oof['true_model'], oof['predicted_model'], class_order))
display(fold_results)
display(class_results.sort_values('recall').reset_index(drop=True))


In [ ]:
from ntl.submission import fit_full_model, predict_test, build_submission, write_and_verify_submission

full_model = fit_full_model(make_model, train)
predictions, decision_scores = predict_test(full_model, test)
submission = build_submission(test, predictions, class_order)
submission_path = RUN_DIR / 'submission.parquet'
write_and_verify_submission(submission, test, class_order, submission_path, [DATA_DIR])
print('Submission validated:', submission_path)
display(submission.head())
